# 04b. Avaliar nota baixa

Depois do [`04_atribuir.ipynb`](04_atribuir.ipynb). Universo: Censo de aplicação
que **não** entrou em `splink_atribuicao`. Desses, o melhor par com
`0,25 ≤ p < 0,5` (parquet do [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb)).

A faixa já passou no blocking. O que importa é o nível de comparação de nome
e de data: regra que já existe e pesa pouco, ou ELSE com padrão repetido.
Quem não tem nem `p ≥ 0,25` é candidato a blocking novo.


In [1]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO_BAIXA = 0.25
TETO_BAIXA = 0.5

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print('Faixa:', PISO_BAIXA, '<= p <', TETO_BAIXA)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

In [2]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability,
    match_weight
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo, unique_id_cpf, match_probability, degrau, regra
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
n_atrib = con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]
print('Pares no parquet:', f'{n_pred:,}')
print('Atribuições:', f'{n_atrib:,}')


Pares no parquet: 6,598,211
Atribuições: 3,661,469


Censo de aplicação sem linha no 04. Desses, o melhor par na faixa
`[0,25, 0,5)`. Quem não tem par nem com `p ≥ 0,25` fica de fora da amostra
de níveis — é o buraco de blocking.


In [3]:
con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
    SELECT 1 FROM atribuicao_2 a
    WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_nota_baixa AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    p.match_weight
FROM splink_predictions p
JOIN censo_sem_cpf s ON s.unique_id = p.unique_id_censo
WHERE p.match_probability >= {PISO_BAIXA}
  AND p.match_probability < {TETO_BAIXA}
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY p.match_probability DESC, p.unique_id_cpf
) = 1
''')

display(con.execute('''
SELECT
    n_sem_cpf,
    n_com_faixa,
    n_sem_cpf - n_com_faixa AS n_sem_p025,
    ROUND(100.0 * n_com_faixa / NULLIF(n_sem_cpf, 0), 1) AS pct_com_faixa,
    ROUND(100.0 * (n_sem_cpf - n_com_faixa) / NULLIF(n_sem_cpf, 0), 1)
        AS pct_sem_p025
FROM (
    SELECT
        (SELECT COUNT(*) FROM censo_sem_cpf) AS n_sem_cpf,
        (SELECT COUNT(*) FROM melhor_nota_baixa) AS n_com_faixa
)
''').df())


,n_sem_cpf,n_com_faixa,n_sem_p025,pct_com_faixa,pct_sem_p025
0,2417561,118838,2298723,4.9,95.1


Níveis iguais aos do treino (`02_treinar`), na mesma ordem. Nome e data no
melhor par da faixa.


In [4]:
split_c = "string_split(ca.nome_completo_phon, ' ')"
split_p = "string_split(pb.nome_completo_phon, ' ')"
n_min = f'least(len({split_c}), len({split_p}))'
prefixo_sql = (
    f'{n_min} >= 2 AND len({split_c}) <> len({split_p}) AND '
    f'list_slice({split_c}, 1, {n_min}) = list_slice({split_p}, 1, {n_min})'
)
jw_ultimo_095_sql = (
    f"{split_c}[-1] = {split_p}[-1] AND "
    'jaro_winkler_similarity(ca.nome_completo_phon, pb.nome_completo_phon) >= 0.95'
)
jw_ultimo_092_sql = (
    f"{split_c}[-1] = {split_p}[-1] AND "
    'jaro_winkler_similarity(ca.nome_completo_phon, pb.nome_completo_phon) >= 0.92'
)
dl_completo_1_sql = (
    'damerau_levenshtein(ca.nome_completo_phon, pb.nome_completo_phon) <= 1'
)
n_tok = f'len({split_c})'
mesmo_n = f'{n_tok} = len({split_p}) AND {n_tok} >= 3'
zip_tok = f'list_zip({split_c}, {split_p})'
dif_count = (
    'list_aggregate(list_transform('
    f'{zip_tok}, '
    "x -> CASE WHEN x[1] <> x[2] THEN 1 ELSE 0 END"
    "), 'sum')"
)
dif_max_dl = (
    'list_aggregate(list_transform('
    f'{zip_tok}, '
    "x -> CASE WHEN x[1] <> x[2] "
    "THEN damerau_levenshtein(x[1], x[2]) ELSE 0 END"
    "), 'max')"
)
um_token_sql = (
    f'{mesmo_n} AND {split_c}[-1] = {split_p}[-1] '
    f'AND {dif_count} = 1 AND {dif_max_dl} <= 2'
)
dois_tokens_sql = (
    f'{mesmo_n} AND {dif_count} BETWEEN 1 AND 2 AND {dif_max_dl} <= 2'
)
n_c = f'len({split_c})'
n_p = f'len({split_p})'
ordem_sql = (
    f'{n_c} = {n_p} AND {n_c} >= 3 '
    f'AND list_sort({split_c}) = list_sort({split_p}) '
    f'AND {split_c} <> {split_p}'
)

ors_token = []
for k in range(1, 11):
    if k == 1:
        sem_c = f'list_slice({split_c}, 2, {n_c})'
        sem_p = f'list_slice({split_p}, 2, {n_p})'
    else:
        sem_c = (
            f'list_concat(list_slice({split_c}, 1, {k - 1}), '
            f'list_slice({split_c}, {k + 1}, {n_c}))'
        )
        sem_p = (
            f'list_concat(list_slice({split_p}, 1, {k - 1}), '
            f'list_slice({split_p}, {k + 1}, {n_p}))'
        )
    ors_token.append(f'({n_c} >= {k} AND {sem_c} = {split_p})')
    ors_token.append(f'({n_p} >= {k} AND {sem_p} = {split_c})')
token_a_mais_sql = (
    f'abs({n_c} - {n_p}) = 1 AND least({n_c}, {n_p}) >= 3 AND ('
    + ' OR '.join(ors_token)
    + ')'
)

null_data_sql = (
    '(ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL) AND '
    '(ca.mes_nascimento IS NULL OR pb.mes_nascimento IS NULL OR '
    'ca.dia_nascimento IS NULL OR pb.dia_nascimento IS NULL)'
)
troca_mes_dia_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.mes_nascimento = pb.dia_nascimento AND '
    'ca.dia_nascimento = pb.mes_nascimento AND '
    'ca.mes_nascimento <> ca.dia_nascimento'
)
dia_diferente_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.mes_nascimento = pb.mes_nascimento AND '
    'ca.dia_nascimento <> pb.dia_nascimento'
)
mes_diferente_sql = (
    'ca.ano_nascimento = pb.ano_nascimento AND '
    'ca.dia_nascimento = pb.dia_nascimento AND '
    'ca.mes_nascimento <> pb.mes_nascimento'
)
ano_pm1_sql = (
    'ca.mes_nascimento = pb.mes_nascimento AND '
    'ca.dia_nascimento = pb.dia_nascimento AND '
    'abs(try_cast(ca.ano_nascimento AS INTEGER) - '
    'try_cast(pb.ano_nascimento AS INTEGER)) = 1'
)

con.execute(f'''
CREATE OR REPLACE TABLE nota_baixa_niveis AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    m.match_weight,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    CASE
        WHEN ca.nome_completo_phon IS NULL OR pb.nome_completo_phon IS NULL
            THEN 'null'
        WHEN ca.nome_completo_phon = pb.nome_completo_phon
            THEN 'exato'
        WHEN {prefixo_sql}
            THEN 'prefixo'
        WHEN {um_token_sql}
            THEN 'um_token_dl2'
        WHEN {dois_tokens_sql}
            THEN 'dois_tokens_dl2'
        WHEN {token_a_mais_sql}
            THEN 'token_a_mais'
        WHEN {ordem_sql}
            THEN 'ordem'
        WHEN {jw_ultimo_095_sql}
            THEN 'jw_095'
        WHEN {jw_ultimo_092_sql}
            THEN 'jw_092'
        WHEN {dl_completo_1_sql}
            THEN 'dl_1'
        ELSE 'else'
    END AS nivel_nome,
    CASE
        WHEN {null_data_sql}
            THEN 'null'
        WHEN ca.data_nascimento = pb.data_nascimento
            THEN 'exato'
        WHEN {troca_mes_dia_sql}
            THEN 'mes_dia_trocados'
        WHEN {dia_diferente_sql}
            THEN 'dia_diferente'
        WHEN {mes_diferente_sql}
            THEN 'mes_diferente'
        WHEN {ano_pm1_sql}
            THEN 'ano_pm1'
        WHEN damerau_levenshtein(ca.data_nascimento, pb.data_nascimento) <= 1
            THEN 'dl_1'
        WHEN damerau_levenshtein(ca.data_nascimento, pb.data_nascimento) <= 2
            THEN 'dl_2'
        ELSE 'else'
    END AS nivel_data
FROM melhor_nota_baixa m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
''')
print('Pares classificados:', con.execute('SELECT COUNT(*) FROM nota_baixa_niveis').fetchone()[0])


Pares classificados: 118838


In [5]:
display(con.execute('''
SELECT
    nivel_nome,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(match_weight), 2) AS avg_w
FROM nota_baixa_niveis
GROUP BY 1
ORDER BY n DESC
''').df())


,nivel_nome,n,pct,avg_p,avg_w
0,exato,83713,70.4,0.372,-0.77
1,um_token_dl2,20736,17.4,0.410,-0.53
2,jw_092,11381,9.6,0.456,-0.25
3,dois_tokens_dl2,2425,2.0,0.429,-0.42
4,jw_095,300,0.3,0.422,-0.46
5,dl_1,283,0.2,0.424,-0.45


In [6]:
display(con.execute('''
SELECT
    nivel_data,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(match_weight), 2) AS avg_w
FROM nota_baixa_niveis
GROUP BY 1
ORDER BY n DESC
''').df())


,nivel_data,n,pct,avg_p,avg_w
0,null,38890,32.7,0.366,-0.81
1,dl_2,38226,32.2,0.371,-0.78
2,dl_1,11201,9.4,0.457,-0.26
3,exato,11184,9.4,0.457,-0.25
4,ano_pm1,10050,8.5,0.387,-0.67
5,dia_diferente,6162,5.2,0.384,-0.70
6,mes_diferente,2704,2.3,0.398,-0.61
7,mes_dia_trocados,421,0.4,0.419,-0.48


In [7]:
display(con.execute('''
SELECT
    nivel_nome,
    nivel_data,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM nota_baixa_niveis
GROUP BY 1, 2
ORDER BY n DESC
LIMIT 20
''').df())


,nivel_nome,nivel_data,n,pct
0,exato,null,38890,32.7
1,exato,dl_2,29858,25.1
2,jw_092,exato,11184,9.4
3,um_token_dl2,dl_2,8016,6.7
4,um_token_dl2,ano_pm1,6246,5.3
5,um_token_dl2,dl_1,6226,5.2
6,exato,dia_diferente,6162,5.2
7,exato,dl_1,3739,3.1
8,exato,mes_diferente,2704,2.3
9,exato,ano_pm1,2187,1.8


Amostra do ELSE do nome e dos níveis do meio com mais volume. Olhar se o
padrão já tem regra (peso baixo) ou se falta nível.


In [8]:
display(con.execute('''
SELECT
    match_probability,
    nivel_nome,
    nivel_data,
    nome_censo,
    nome_cpf,
    dob_censo,
    dob_cpf
FROM nota_baixa_niveis
WHERE nivel_nome = 'else'
ORDER BY match_probability DESC
LIMIT 40
''').df())


,match_probability,nivel_nome,nivel_data,nome_censo,nome_cpf,dob_censo,dob_cpf


In [9]:
display(con.execute('''
SELECT
    match_probability,
    nivel_nome,
    nivel_data,
    nome_censo,
    nome_cpf,
    dob_censo,
    dob_cpf
FROM nota_baixa_niveis
WHERE nivel_nome IN (
    'prefixo', 'um_token_dl2', 'dois_tokens_dl2', 'token_a_mais',
    'ordem', 'jw_095', 'jw_092', 'dl_1'
)
ORDER BY match_probability DESC
LIMIT 40
''').df())


,match_probability,nivel_nome,nivel_data,nome_censo,nome_cpf,dob_censo,dob_cpf
0,0.499752,jw_092,exato,KAIKI KOSTA SIUVA,KAIKI ROXA SIUVA,2017-12-31,2017-12-31
1,0.499752,jw_092,exato,BRUNU JEAKOMI MILOMEN MARINHU,BRUNU JIAKOMELI MILOMEN MARINHU,1981-08-22,1981-08-22
2,0.499752,jw_092,exato,MARIONEIDI KOSTA MONTEIRU,MARIA ONEIDI KOSTA MONTEIRU,1983-01-12,1983-01-12
3,0.499752,jw_092,exato,ANA ALISI KOSTA,ANALISI KOSTA,1975-11-22,1975-11-22
4,0.499752,jw_092,exato,DOMINGUS ERIKI KOSTA SIUVA,DOMINGUS ERIKI LOPIS SIUVA,2009-12-05,2009-12-05
5,0.499752,jw_092,exato,UELINTON KARDOZU KOSTA,UELINGUITON KARDOZU KOSTA,1989-08-21,1989-08-21
6,0.499752,jw_092,exato,JOZI ADAIUTON PEREIRA MENEZIS,JOZI ADAIU PEREIRA MENEZIS,1974-11-04,1974-11-04
7,0.499752,jw_092,exato,EURI MAIORON LIMA FONSEKA,EURIMAIRON LIMA FONSEKA,1987-02-15,1987-02-15
8,0.499752,jw_092,exato,JOZI DIONI SIUVA SOUZA,JOZI JONIS SIUVA SOUZA,1988-12-30,1988-12-30
9,0.499752,jw_092,exato,LUNA ARAUJU,LUNA GOMIS ARAUJU,2020-08-25,2020-08-25


In [10]:
con.close()